In [2]:
import pandas as pd
import json
from typing import Any
import re

In [84]:
def format_dataset(dataset: list[dict[str, Any]]) -> None:
    def format_data(data: dict[str, Any], unique_id: int) -> None:
        def insert_tags(s, word, tag):
            return re.sub(re.escape(word), f"<{tag}>{word}</{tag}>", s, count=1)
        
        sentence_list = data['Source sentence'].split('</s></s>')
        if len(sentence_list) < 3:
            print(data['Source sentence'])
            
        input_text_no_tag = sentence_list[-1].strip()
        data['input_text_no_tag'] = input_text_no_tag
        
        source = data['Event1'].strip()
        target = data['Event2'].strip()
        data['source'] = source
        data['target'] = target
        
        # 找到较早出现的事件，以便先处理
        idx1 = input_text_no_tag.find(source)
        idx2 = input_text_no_tag.find(target)
    
        if idx1 < idx2:
            input_text_no_tag = insert_tags(input_text_no_tag, source, 'event1')
            input_text_no_tag = insert_tags(input_text_no_tag, target, 'event2')
        else:
            input_text_no_tag = insert_tags(input_text_no_tag, target, 'event2')
            input_text_no_tag = insert_tags(input_text_no_tag, source, 'event1')
        data['input_text'] = input_text_no_tag  # 字符串，插入<event1></event1>和<event2></event2>
        
        
        data['ground'] = 1 if data['labels'] == 'Cause-Effect' else 0
        data['unique_id'] = unique_id
        
    for idx, data in enumerate(dataset):
        format_data(data, idx)

## CTB

In [ ]:
split = 'test'  # 'test' and 'train'

ctb_dataset_original_path = r'dataset\Code4CPATT\data\{split}_causal_timebank.csv'.format(split=split)
ctb_dataset_formatted_path = r'dataset\CTB\full_{split}.jsonl'.format(split=split)

ctb_dataset = pd.read_csv(ctb_dataset_original_path)
ctb_dataset_jsonl = json.loads(ctb_dataset.to_json(orient='records', indent=4, force_ascii=False))

format_dataset(ctb_dataset_jsonl)

with open(ctb_dataset_formatted_path, 'w', encoding='utf-8') as f:
    f.writelines(json.dumps(data, ensure_ascii=False) + '\n' for data in ctb_dataset_jsonl)

## ESC

In [ ]:
split = 'test'  # 'test' and 'train'
subset_type = 'intra'  # 'intra' and 'inter'

esc_dataset_original_path = r'dataset\Code4CPATT\data\{split}_eventstoryline.csv'.format(split=split)
esc_dataset_formatted_path = r'dataset\ESC - {subset_type}\{split}_train.jsonl'.format(subset_type=subset_type, split=split)

esc_dataset = pd.read_csv(esc_dataset_original_path)
esc_dataset = esc_dataset[esc_dataset['type'] == subset_type]
esc_dataset_jsonl = json.loads(esc_dataset.to_json(orient='records', indent=4, force_ascii=False))

format_dataset(esc_dataset_jsonl)

with open(esc_dataset_formatted_path, 'w', encoding='utf-8') as f:
    f.writelines(json.dumps(data, ensure_ascii=False) + '\n' for data in esc_dataset_jsonl)